# Einops 基础教程

参考: https://einops.rocks/1-einops-basics/

## 为什么使用 einops？

传统写法：
```python
y = x.transpose(0, 2, 3, 1)
```

einops 写法（更易读）：
```python
y = rearrange(x, 'b c h w -> b h w c')
```

einops 支持 numpy、pytorch、jax、tensorflow 等主流框架。


In [10]:
# 安装 einops (如果还没安装)
!pip install einops

import numpy as np
import torch
from einops import rearrange, reduce, repeat

print("einops 导入成功！")


Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://mirrors.ustc.edu.cn/pypi/simple
einops 导入成功！


## 1. rearrange - 重排维度

`rearrange` 是最核心的操作，可以实现 transpose、reshape、squeeze、expand_dims 等功能。


In [11]:
# 创建一个示例张量: batch=2, channel=3, height=4, width=5
x = torch.randn(2, 3, 4, 5)
print(f"原始形状: {x.shape}")


原始形状: torch.Size([2, 3, 4, 5])


In [14]:
# 1.1 基本转置: BCHW -> BHWC
y = rearrange(x, 'b c h w -> b h w c')
print(f"转置后: {y.shape}")  # [2, 4, 5, 3]


转置后: torch.Size([2, 4, 5, 3])


In [15]:
# 1.2 合并维度 (Composition)
# 将 batch 和 height 合并
y = rearrange(x, 'b c h w -> (b h) c w')
print(f"合并 b 和 h: {y.shape}")  # [8, 3, 5]


合并 b 和 h: torch.Size([8, 3, 5])


In [7]:
# 1.3 拆分维度 (Decomposition)
# 将 channel=6 拆分为 2x3
x2 = torch.randn(2, 6, 4, 5)
y = rearrange(x2, 'b (c1 c2) h w -> b c1 c2 h w', c1=2)
print(f"拆分 channel: {y.shape}")  # [2, 2, 3, 4, 5]


拆分 channel: torch.Size([2, 2, 3, 4, 5])


In [ ]:
# 1.4 Flatten 操作
y = rearrange(x, 'b c h w -> b (c h w)')
print(f"Flatten: {y.shape}")  # [2, 60]


## 2. reduce - 归约操作

`reduce` 在重排的同时进行归约（mean, max, min, sum, prod）。


In [18]:
# 2.1 全局平均池化 (Global Average Pooling)
x = torch.randn(2, 3, 4, 5)
y = reduce(x, 'b c h w -> b c', 'mean')
print(f"GAP: {y.shape}")  # [2, 3]


GAP: torch.Size([2, 3])


In [20]:
# 2.2 沿 channel 求最大值
y = reduce(x, 'b c h w -> b h w', 'max')
print(f"Max over channel: {y.shape}")  # [2, 4, 5]


Max over channel: torch.Size([2, 4, 5])


In [21]:
# 2.3 下采样 (Pooling)
# 2x2 平均池化
x3 = torch.randn(2, 3, 8, 8)
y = reduce(x3, 'b c (h h2) (w w2) -> b c h w', 'mean', h2=2, w2=2)
print(f"2x2 AvgPool: {y.shape}")  # [2, 3, 4, 4]


2x2 AvgPool: torch.Size([2, 3, 4, 4])


In [22]:
# 2.4 保持维度 (keepdim)
# 使用 () 创建大小为 1 的维度
y = reduce(x, 'b c h w -> b c () ()', 'mean')
print(f"Mean with keepdim: {y.shape}")  # [2, 3, 1, 1]


Mean with keepdim: torch.Size([2, 3, 1, 1])


## 3. repeat - 重复/复制

`repeat` 用于复制/广播张量。


In [ ]:
# 3.1 添加 batch 维度并复制
x = torch.randn(3, 4)
y = repeat(x, 'h w -> b h w', b=5)
print(f"添加 batch: {y.shape}")  # [5, 3, 4]


In [ ]:
# 3.2 沿某个维度复制
y = repeat(x, 'h w -> h (tile w)', tile=3)
print(f"沿 w 复制 3 次: {y.shape}")  # [3, 12]


In [ ]:
# 3.3 上采样 (Upsampling)
x4 = torch.randn(2, 3, 4, 4)
y = repeat(x4, 'b c h w -> b c (h h2) (w w2)', h2=2, w2=2)
print(f"2x 上采样: {y.shape}")  # [2, 3, 8, 8]


## 4. 实际应用示例

下面展示 einops 在深度学习中的常见用法。


In [ ]:
# 4.1 Multi-Head Attention 中的 reshape
# 输入: (batch, seq_len, d_model)
# 输出: (batch, num_heads, seq_len, d_k)

batch, seq_len, d_model, num_heads = 2, 10, 64, 8
x = torch.randn(batch, seq_len, d_model)

# 传统写法
# y = x.view(batch, seq_len, num_heads, d_model // num_heads).transpose(1, 2)

# einops 写法 (更清晰!)
y = rearrange(x, 'b s (h d) -> b h s d', h=num_heads)
print(f"Multi-head reshape: {y.shape}")  # [2, 8, 10, 8]


In [ ]:
# 4.2 Vision Transformer 的 Patch Embedding
# 将图像切分为 patches

batch, channels, height, width = 2, 3, 224, 224
patch_size = 16
img = torch.randn(batch, channels, height, width)

# 切分为 patches 并展平
patches = rearrange(img, 'b c (h p1) (w p2) -> b (h w) (p1 p2 c)', p1=patch_size, p2=patch_size)
print(f"Patches: {patches.shape}")  # [2, 196, 768]  (196 = 14*14 patches, 768 = 16*16*3)


In [ ]:
# 4.3 RMSNorm 中计算均方根
x = torch.randn(2, 10, 64)  # (batch, seq, d_model)

# 计算最后一维的均方根
rms = reduce(x ** 2, 'b s d -> b s ()', 'mean').sqrt()
print(f"RMS shape: {rms.shape}")  # [2, 10, 1]

# 归一化
x_norm = x / (rms + 1e-6)
print(f"Normalized: {x_norm.shape}")


## 5. 常用模式速查

| 操作 | einops 写法 |
|------|-------------|
| transpose | `rearrange(x, 'b c h w -> b h w c')` |
| flatten | `rearrange(x, 'b c h w -> b (c h w)')` |
| squeeze | `rearrange(x, 'b 1 h w -> b h w')` |
| unsqueeze | `rearrange(x, 'b h w -> b 1 h w')` |
| global avg pool | `reduce(x, 'b c h w -> b c', 'mean')` |
| 2x2 avg pool | `reduce(x, 'b c (h 2) (w 2) -> b c h w', 'mean')` |
| tile/repeat | `repeat(x, 'h w -> (tile h) w', tile=3)` |
| multi-head reshape | `rearrange(x, 'b s (h d) -> b h s d', h=num_heads)` |


In [ ]:
print("🎉 教程完成！现在你可以在 Transformer 实现中使用 einops 了。")
